In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1단계 : 데이터 준비
price = yf.download([
    'AAPL',
    'MSFT',
    'GOOGL',
    'JNJ',
], 
 start='2022-01-01', 
 auto_adjust='True',
progress=False)['Close']

tickers = price.columns
assets_count = len(tickers)

# 2단계 : 일별 수익률 
returns = price.pct_change(fill_method=None).dropna()

# 3단계  : 평균 수익률과 공분산
mean_returns = returns.mean() * 252
cov_matrix= returns.cov() * 252

# 4단계 : 최고의 조합 찾기
rf = 0.02                # 무위험수익률 (예: 2%)
best = None  

for _ in range(20000):
    w = np.random.random(assets_count)
    w = w/w.sum()

    port_return = np.dot(w, mean_returns)
    port_risk = np.sqrt(w @ cov_matrix @ w)   
    sharpe = (port_return - rf) / port_risk

    if best is None or sharpe > best['sharpe']:
        best = {'sharpe': sharpe, 'return': port_return,
                'risk': port_risk, 'weights': w}




# 5단계 : 결과
print(f"최고 샤프지수 : {best['sharpe']:.2f}")
print(f"기대 수익률 : {best['return']:.1%}, 위험: {best['risk']:.1%}")
print("─ 추천 비중 ─")
for ticker, weight in zip(tickers, best['weights']):
    print(f"  {ticker}: {weight:.1%}")